In [ ]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

# Load environment variables (ensure GROQ_API_KEY is set in your .env)
load_dotenv(find_dotenv())

# Initialize Groq model
free_model = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0.7,
)

agent = create_agent(
    model=free_model,
    checkpointer=InMemorySaver(), # InMemorySaver remembers the conversation history across turns for your thread_id. Without it, every call to agent.invoke() would start with an empty history and never reach 10 messages.
    middleware=[
        SummarizationMiddleware(
            model=free_model,
            trigger=("messages", 10), # when message length is greater then 10 then trigger the Summarization
            keep=("messages", 4), # keep recent top 4 message
        )
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Q: {q} | {response['messages'][0].content} |  Messages in state: {len(response['messages'])}") # Item count: len(response['messages']) returns how many total messages exist in the history (e.g., 2, 4, 6). Each user question and AI response counts as 1 separate message object in that list.

Q: What is 2+2? | What is 2+2? |  Messages in state: 2
Q: What is 10*5? | What is 2+2? |  Messages in state: 4
Q: What is 100/4? | What is 2+2? |  Messages in state: 6
Q: What is 15-7? | What is 2+2? |  Messages in state: 8
Q: What is 3*3? | What is 2+2? |  Messages in state: 10
Q: What is 4*4? | Here is a summary of the conversation to date:

## SESSION INTENT
The user is requesting solutions to a series of simple arithmetic problems.

## SUMMARY
The user is asking sequential basic math questions. The following calculations have been completed and answered:
- 2 + 2 = 4
- 10 × 5 = 50
- 100 ÷ 4 = 25

The current pending question is "What is 15-7?".

## ARTIFACTS
None

## NEXT STEPS
1. Answer the current question: 15 - 7 = 8.
2. Await the next arithmetic problem from the user. |  Messages in state: 6


In [15]:
from openai.types.beta import beta_response_code_interpreter_call_interpreting_event
from openai.types.beta import beta_response_code_interpreter_call_in_progress_event
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"
    
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"
    
agent = create_agent(
        model=free_model,
        tools=[read_email_tool, send_email_tool],
        checkpointer=InMemorySaver(),
        middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
        "read_email_tool": False, # never needs approval
        })],)

config = {"configurable": {"thread_id": "email-session-1"}}        

In [17]:
# --- EXECUTION FLOW ---

# Step 1: Prompt the agent to take an action requiring approval
print("--- Invoking Agent ---")
response = agent.invoke(
    {"messages": [HumanMessage(content="Send an email to manager@company.com with subject 'Status' and body 'Completed all tasks.')")]},
    config=config,
)

# The agent intercepts before executing send_email_tool and pauses
print("\nAgent state hit an interrupt for tool execution approval.")

# Step 2: Resume execution with a human decision ('approve', 'edit', or 'reject')
# Step 2: Resume execution with a human decision
print("\n--- Human Approves Action ---")
final_response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=config,
)

print("\nFinal Output:")
print(final_response["messages"][-1].content)

--- Invoking Agent ---

Agent state hit an interrupt for tool execution approval.

--- Human Approves Action ---

Final Output:
The email has been sent to manager@company.com with the subject "Status" and body "Completed all tasks."


In [18]:
from langgraph.types import Command

# Resume execution with modified parameters
final_response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email_tool",
                        "args": {
                            "recipient": "lead@company.com",  # Changed recipient
                            "subject": "Updated Task Status",  # Changed subject
                            "body": "All high-priority tasks completed.",  # Changed body
                        },
                    },
                }
            ]
        }
    ),
    config=config,
)

print(final_response["messages"][-1].content)

The email has been sent to manager@company.com with the subject "Status" and body "Completed all tasks."


In [19]:
from langgraph.types import Command

# Resume execution by rejecting the action
final_response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "Action cancelled: Recipient email address is incorrect.",
                }
            ]
        }
    ),
    config=config,
)

print(final_response["messages"][-1].content)

The email has been sent to manager@company.com with the subject "Status" and body "Completed all tasks."
